In [1]:
import osmnx as ox         # Downloads, models, and analyzes street networks from OpenStreetMap
import networkx as nx      # Creates and analyzes graph structures; used for shortest-path calculations
from shapely.geometry import Point     # Represents geographic points using coordinates (longitude, latitude)
import folium              # Creates interactive maps and visualizations
import webbrowser          # Opens generated HTML files in the default web browser
import os                  # Handles file paths and operating system interactions
import pyproj              # Performs cartographic projections and geodetic/ellipsoidal calculations 
import pandas as pd        # Provides data structures and tools for data analysis and manipulation

In [2]:
def calculate_geodesic_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the high-precision geodesic distance between two points 
    on the WGS84 ellipsoidal surface using pyproj.
    """
    # Load standard WGS84 ellipsoid
    geod = pyproj.Geod(ellps="WGS84")
    
    # pyproj expects coordinates in (Longitude, Latitude) order
    _, _, dist_meters = geod.inv(lon1, lat1, lon2, lat2)
    
    # Convert meters to kilometers
    return dist_meters / 1000.0

In [3]:
def calculate_shortest_road_path(start_coords, end_coords,graph,graph_proj):

    # Extract latitude and longitude from the start and end coordinates
    lat1, lon1 = start_coords[0], start_coords[1]
    lat2, lon2 = end_coords[0], end_coords[1]  

    # Calculate the straight-line (geodesic) distance as a fallback
    geodesic_dis=calculate_geodesic_distance(lat1, lon1, lat2, lon2)

    try:
    # Create Shapely Point objects using (longitude, latitude)
     point1 = Point(lon1, lat1)
     point2 = Point(lon2, lat2)
    
    # Project points into the same coordinate reference system (CRS) as the projected road network graph
     point1_proj, _ = ox.projection.project_geometry(point1, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
     point2_proj, _ = ox.projection.project_geometry(point2, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
    
    # Find the nearest road network nodes to the projected points
     start_node = ox.nearest_nodes(graph_proj, X=point1_proj.x, Y=point1_proj.y) 
     end_node = ox.nearest_nodes(graph_proj, X=point2_proj.x, Y=point2_proj.y)

    # Compute the shortest road path based on edge length
     shortest_route = nx.shortest_path(graph_proj, start_node, end_node, weight="length") 
    
    # Convert the route into a GeoDataFrame and calculate total road distance
     gdf_edges = ox.routing.route_to_gdf(graph_proj, shortest_route)
     actual_road_km = gdf_edges["length"].sum() / 1000 
    
    # Extract route coordinates for map visualization
     route_coordinates = []
     for node in shortest_route:
        node_data = graph.nodes[node]
        route_coordinates.append((node_data['y'], node_data['x']))

    # Return road distance and route coordinates    
     return actual_road_km, route_coordinates
    
    except nx.NetworkXNoPath:
        # Handle cases where no road connection exists between locations
        print("No road path exists between these locations, using geodesic distance.")
        return geodesic_dis,[]
    
    except Exception as e:
        # Handle routing or projection errors and fall back to geodesic distance
        print(f"Road routing unavailable: {e}")
        # Fallback to geodesic distance
        return geodesic_dis,[]


In [4]:
def process_table_and_map(df,graph,graph_proj):
    # Initialize total trip distance accumulator
    total_trip_distance = 0.0

    # Get the starting location from the first row of the dataframe
    init_lat = (df['latitude'].max()+df['latitude'].min())/2
    init_lon = (df['longitude'].max()+df['longitude'].min())/2

     # Google Maps tile layer URL for the Folium map
    google_tiles_url = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}" 

     # Create a Folium map centered on the first location
    map_point = folium.Map(
       location=[init_lat, init_lon], 
       zoom_start=11,
       tiles=google_tiles_url, 
       attr='Google'
    )
    # Add a marker for the trip starting point
    folium.Marker(
        location=[init_lat, init_lon],
        tooltip="Start Location (Point 1)",
        icon=folium.Icon(color='red')
    ).add_to(map_point)

    # Iterate through consecutive pairs of locations
    for i in range(len(df) - 1):

        # Current point and next point coordinates
        start_pt = (df.loc[i, 'latitude'], df.loc[i, 'longitude'])
        end_pt = (df.loc[i+1, 'latitude'], df.loc[i+1, 'longitude'])

        # Calculate shortest road path and distance between points
        segment_km, route_coords = calculate_shortest_road_path(start_pt, end_pt,graph,graph_proj)

        # Print whether road-network distance or geodesic distance was used
        if route_coords and len(route_coords) > 0:
            print(f"Actual distance between Point {i+1} and Point {i+2} is:{segment_km:.2f} km")
        else:
            print(f"Geodesic distance between Point {i+1} and Point {i+2} is:{segment_km:.2f} km")
        
        # Add segment distance to total trip distance
        total_trip_distance += segment_km

        # Add marker for the destination point of the current segment
        folium.Marker(
            location=end_pt,
            tooltip=f"Stop {i+2}. Leg Distance: {segment_km:.2f} km",
            icon=folium.Icon(color='red')
        ).add_to(map_point)

        # Draw route on map
        if route_coords and len(route_coords) > 0:
         # Draw actual road path if route coordinates are available
         folium.PolyLine(
            locations=route_coords, 
            color="blue", 
            weight=5, 
            opacity=0.8,
            popup=f"Leg {i+1}: {segment_km:.2f} km"
         ).add_to(map_point)
        else:
         # Draw straight-line path when road route cannot be found
         folium.PolyLine(
            locations=[start_pt, end_pt],
            color="gray",
            weight=3,
            opacity=0.6,
            dash_array="5,5",
            popup=f"Leg {i+1}: {segment_km:.2f} km (geodesic)"
         ).add_to(map_point)

    # Display total cumulative distance of the trip
    print(f"Total cumulative road distance across all stops: {total_trip_distance:.2f} km")

    # Save map as an HTML file
    file_name = "dataframe_distance_mapping.html"
    map_point.save(file_name)  

    # Open the generated map in the default web browser             
    webbrowser.open('file://' + os.path.realpath(file_name)) 

In [5]:
if __name__ == "__main__":
    # Read pincode/location data from Excel file
    data=pd.read_excel('Pincode_table.xlsx')

    print("------------------------------ORIGINAL DATAFRAME------------------------------\n")
    display(data)
    
    print("\n------------------------------DATAFRAME INFO------------------------------\n")
    data.info()

    # Filter records belonging to Maharashtra state
    data=data[data['statename'].str.contains("Maharashtra", case=False,na=False)]

    print("\n------------------------------AFTER MAHARASHTRA FILTER------------------------------\n")
    display(data)
    print("\n------------------------------DATASET INFO AFTER MAHARASHTRA FILTER------------------------------\n")
    data.info()

    # Replace missing latitude and longitude values with 0
    data[['latitude', 'longitude']] = data[['latitude', 'longitude']].fillna(0)

    print("\n------------------------------DATASET INFO AFTER HANDLING NULL VALUES------------------------------\n")
    data.info()

    # Filter records belonging to the Mumbai region
    data=data[data['regionname'].str.contains('Mumbai',case=False,na=False)]

    # Further filter records belonging to Mumbai district
    data=data[data['district'].str.contains('Mumbai',case=False,na=False)]

    print("\n------------------------------AFTER MUMBAI REGION AND MUMBAI DISTRICT FILTER------------------------------\n")
    display(data)
     
    print("\n------------------------------DATASET INFO AFTER MUMBAI REGION AND MUMBAI DISTRICT FILTER------------------------------\n")
    data.info()

    # Convert latitude and longitude columns to numeric values
    # Invalid values will be converted to NaN
    data['latitude'] = pd.to_numeric(data['latitude'], errors='coerce')
    data['longitude'] = pd.to_numeric(data['longitude'], errors='coerce')

    print("\n----------------------------DATASET INFO AFTER CHANGING DATATYPES OF LATITUDES AND LONGITUDES----------------------------\n")
    data.info()
    
    data.reset_index(drop=True, inplace=True)

    invalid_rows1=data[data['latitude']>20]
    invalid_rows2=data[data['longitude']>73]
    invalid_rows3=data[data['latitude']<18]
    invalid_rows4=data[data['longitude']<71]
    print("\n------------------------------INVALID ROWS------------------------------\n")
    display(invalid_rows1)
    display(invalid_rows2)
    display(invalid_rows3)
    display(invalid_rows4)


    # Remove a specific problematic/outlier record by index
    data=data.drop(data.index[227])
    data=data.drop(data.index[187])

    print("\n------------------------------DATASET INFO AFTER REMOVING INVALID ROWS------------------------------\n")
    data.info()

    # Reset dataframe index after row removal
    data.reset_index(drop=True, inplace=True)

    print("\n------------------------------FINAL DATAFRAME------------------------------\n")
    display(data)          # Display dataframe 

    # to ensure road network coverage extends beyond data points
    padding = 0.03                                                 

    # Determine geographic bounding box limits
    north =data["latitude"].max() + padding
    south =data["latitude"].min() - padding
    east =data["longitude"].max() + padding
    west =data["longitude"].min() - padding 
    
    # Download drivable road network within the bounding box
    graph = ox.graph_from_bbox(bbox=(west, south, east, north), network_type="drive")

    # Verify that a valid road network was retrieved
    if len(graph) == 0:
        raise ValueError("No drivable roads found in this specific coordinate region.")
    
    # Project graph to a suitable coordinate reference system for accurate distance calculations
    graph_proj = ox.project_graph(graph)
    
    # Calculate distances, generate route map,and display/save the final output
    process_table_and_map(data,graph,graph_proj)

------------------------------ORIGINAL DATAFRAME------------------------------



,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
0,Telangana Circle,Hyderabad Region,Adilabad Division,Kothimir B.O,504273,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.363869,79.537666
1,Telangana Circle,Hyderabad Region,Adilabad Division,Papanpet B.O,504299,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.47649,79.583923
2,Telangana Circle,Hyderabad Region,Adilabad Division,Kukuda B.O,504299,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,NaN,NaN
3,Telangana Circle,Hyderabad Region,Adilabad Division,Bareguda B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.328575,79.476013
4,Telangana Circle,Hyderabad Region,Adilabad Division,Mosam B.O,504296,BO,Delivery,KUMURAM BHEEM ASIFABAD,TELANGANA,19.377804,79.616521
...,...,...,...,...,...,...,...,...,...,...,...
165622,West Bengal Circle,South Bengal Region,Tamluk Division,Raghunathbari SO,721634,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.353663,87.797606
165623,West Bengal Circle,South Bengal Region,Tamluk Division,Ramchandrapur SO East Midnapore,721647,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.2975,87.7542
165624,West Bengal Circle,South Bengal Region,Tamluk Division,Srirampur Mid SO,721651,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.303121,87.913554
165625,West Bengal Circle,South Bengal Region,Tamluk Division,Gopinathpur SO,721638,PO,Delivery,MEDINIPUR EAST,WEST BENGAL,22.112991,87.799122



------------------------------DATAFRAME INFO------------------------------

<class 'pandas.DataFrame'>
RangeIndex: 165627 entries, 0 to 165626
Data columns (total 11 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   circlename    165627 non-null  str   
 1   regionname    165312 non-null  str   
 2   divisionname  165627 non-null  str   
 3   officename    165627 non-null  str   
 4   pincode       165627 non-null  int64 
 5   officetype    165627 non-null  str   
 6   delivery      165627 non-null  str   
 7   district      164912 non-null  str   
 8   statename     164912 non-null  str   
 9   latitude      153620 non-null  object
 10  longitude     153625 non-null  object
dtypes: int64(1), object(2), str(8)
memory usage: 13.9+ MB

------------------------------AFTER MAHARASHTRA FILTER------------------------------



,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
10837,Maharashtra Circle,Goa-Panaji Region,Kolhapur Division,Kapshi B.O,416214,BO,Delivery,KOLHAPUR,MAHARASHTRA,NaN,NaN
10838,Maharashtra Circle,Goa-Panaji Region,Kolhapur Division,Rethare B.O,416214,BO,Delivery,KOLHAPUR,MAHARASHTRA,NaN,NaN
10839,Maharashtra Circle,Goa-Panaji Region,Kolhapur Division,Ukhalu B.O,416214,BO,Delivery,KOLHAPUR,MAHARASHTRA,NaN,NaN
10840,Maharashtra Circle,Goa-Panaji Region,Kolhapur Division,Molawade B.O,416215,BO,Delivery,KOLHAPUR,MAHARASHTRA,NaN,NaN
10841,Maharashtra Circle,Goa-Panaji Region,Kolhapur Division,Shankarwadi B.O,416232,BO,Delivery,KOLHAPUR,MAHARASHTRA,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
163483,Maharashtra Circle,Regional Office Aurangabad,Pharbhani Division,Bori S.O (Parbhani),431508,PO,Delivery,PARBHANI,MAHARASHTRA,19.493595,76.731658
163484,Maharashtra Circle,Regional Office Aurangabad,Pharbhani Division,Mau Parbhani S.O,431402,PO,Delivery,PARBHANI,MAHARASHTRA,19.249426,76.7818
163485,Maharashtra Circle,Regional Office Aurangabad,Pharbhani Division,Naya Mondha Parbhani S.O,431401,PO,Non Delivery,PARBHANI,MAHARASHTRA,19.264528,76.768774
163486,Maharashtra Circle,Regional Office Aurangabad,Pharbhani Division,Station Parbhani S.O,431401,PO,Non Delivery,PARBHANI,MAHARASHTRA,19.259479,76.772976



------------------------------DATASET INFO AFTER MAHARASHTRA FILTER------------------------------

<class 'pandas.DataFrame'>
Index: 13762 entries, 10837 to 163487
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   circlename    13762 non-null  str   
 1   regionname    13762 non-null  str   
 2   divisionname  13762 non-null  str   
 3   officename    13762 non-null  str   
 4   pincode       13762 non-null  int64 
 5   officetype    13762 non-null  str   
 6   delivery      13762 non-null  str   
 7   district      13762 non-null  str   
 8   statename     13762 non-null  str   
 9   latitude      12926 non-null  object
 10  longitude     12925 non-null  object
dtypes: int64(1), object(2), str(8)
memory usage: 1.3+ MB

------------------------------DATASET INFO AFTER HANDLING NULL VALUES------------------------------

<class 'pandas.DataFrame'>
Index: 13762 entries, 10837 to 163487
Data columns (total 11 columns)

...

,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude


,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
227,Maharashtra Circle,Mumbai Region,Mumbai City South Division,High Court Building SO Mumbai,400032,PO,Non Delivery,MUMBAI,MAHARASHTRA,72.8302,18.9311



------------------------------DATASET INFO AFTER REMOVING INVALID ROWS------------------------------

<class 'pandas.DataFrame'>
Index: 227 entries, 0 to 228
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   circlename    227 non-null    str    
 1   regionname    227 non-null    str    
 2   divisionname  227 non-null    str    
 3   officename    227 non-null    str    
 4   pincode       227 non-null    int64  
 5   officetype    227 non-null    str    
 6   delivery      227 non-null    str    
 7   district      227 non-null    str    
 8   statename     227 non-null    str    
 9   latitude      227 non-null    float64
 10  longitude     227 non-null    float64
dtypes: float64(2), int64(1), str(8)
memory usage: 21.3 KB

------------------------------FINAL DATAFRAME------------------------------



,circlename,regionname,divisionname,officename,pincode,officetype,delivery,district,statename,latitude,longitude
0,Maharashtra Circle,Mumbai Region,Mumbai City East Division,Princess Dock S.O,400009,PO,Non Delivery,MUMBAI,MAHARASHTRA,18.950528,72.840833
1,Maharashtra Circle,Mumbai Region,Mumbai City East Division,Naigaon S.O (Mumbai),400014,PO,Non Delivery,MUMBAI,MAHARASHTRA,19.004842,72.846009
2,Maharashtra Circle,Mumbai Region,Mumbai City North East Division,Kurla West SO,400070,PO,Delivery,MUMBAI SUBURBAN,MAHARASHTRA,19.068833,72.877783
3,Maharashtra Circle,Mumbai Region,Mumbai City North East Division,Raoli Camp SO,400022,PO,Non Delivery,MUMBAI,MAHARASHTRA,19.035528,72.866802
4,Maharashtra Circle,Mumbai Region,Mumbai City North East Division,Sion SO,400022,PO,Delivery,MUMBAI,MAHARASHTRA,19.044426,72.863721
...,...,...,...,...,...,...,...,...,...,...,...
222,Maharashtra Circle,Mumbai Region,Mumbai City West Division,Rajbhavan S.O (Mumbai),400035,PO,Delivery,MUMBAI,MAHARASHTRA,18.943465,72.794326
223,Maharashtra Circle,Mumbai Region,Mumbai City West Division,Tardeo S.O,400007,PO,Non Delivery,MUMBAI,MAHARASHTRA,18.998889,72.840083
224,Maharashtra Circle,Mumbai Region,Mumbai City South Division,Colaba Bazar SO,400005,PO,Non Delivery,MUMBAI,MAHARASHTRA,18.910915,72.806482
225,Maharashtra Circle,Mumbai Region,Mumbai City South Division,Colaba SO,400005,PO,Delivery,MUMBAI,MAHARASHTRA,18.911225,72.821007


Actual distance between Point 1 and Point 2 is:6.95 km
Actual distance between Point 2 and Point 3 is:9.46 km
Actual distance between Point 3 and Point 4 is:6.47 km
Actual distance between Point 4 and Point 5 is:1.53 km
Actual distance between Point 5 and Point 6 is:5.74 km
Actual distance between Point 6 and Point 7 is:7.37 km
Actual distance between Point 7 and Point 8 is:2.86 km
Actual distance between Point 8 and Point 9 is:6.18 km
Actual distance between Point 9 and Point 10 is:5.62 km
Actual distance between Point 10 and Point 11 is:3.30 km
Actual distance between Point 11 and Point 12 is:9.92 km
Actual distance between Point 12 and Point 13 is:4.72 km
Actual distance between Point 13 and Point 14 is:4.37 km
Actual distance between Point 14 and Point 15 is:2.13 km
Actual distance between Point 15 and Point 16 is:2.32 km
Actual distance between Point 16 and Point 17 is:10.54 km
Actual distance between Point 17 and Point 18 is:2.97 km
Actual distance between Point 18 and Point 19 i